# 🔬 Notebook Analisis Kritis: AI Adoption & Productivity Analysis (2021–2026)
**Project:** Data Analyst Portfolio - Deep Dive & Critical Insights  
**Dataset Files:** `ai_adoption_productivity_2021_2026.csv` & `user_level_ai_adoption.csv`  
**Data Analyst Author:** Rafli A.  
**Dataset Source:** [Global AI Usage and Productivity - Kaggle](https://www.kaggle.com/datasets/ashyou09/global-ai-usage-and-productivity) 

---

## 📌 Latar Belakang, Sitasi & Tujuan Analisis Kritis
Notebook ini dirancang untuk menyajikan **analisis kritis dan mendalam** terhadap data adopsi AI dan produktivitas pengguna.

> **Dataset Credit & Attribution:**  
> Dataset yang digunakan dalam proyek analisis data ini berasal dari sumber open source:  
> 🔗 [Global AI Usage and Productivity Dataset - Kaggle](https://www.kaggle.com/datasets/ashyou09/global-ai-usage-and-productivity)

Analisis ini tidak hanya menyajikan angka agregat, tetapi juga menguji validitas hipotesis, mendeteksi anomali/paradoks data, mengevaluasi potensi bias *self-reporting*, serta mengukur variabel penentu produktivitas berbasis pemodelan statistik dan *unsupervised clustering*.

### 1. Inisialisasi Environment & Input Data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
import os

# Configuration
plt.style.use('ggplot')
pd.set_option('display.max_columns', None)

# Load Datasets
df_macro = pd.read_csv('../data/ai_adoption_productivity_2021_2026.csv')
df_micro = pd.read_csv('../data/user_level_ai_adoption.csv')

print(f"Macro Dataset Shape: {df_macro.shape}")
print(f"Micro Dataset Shape: {df_micro.shape}")

Macro Dataset Shape: (402, 6)
Micro Dataset Shape: (15000, 10)


### 2. Audit Kualitas Data & Pemeriksaan Distribusi (Data Quality & Anomaly Detection)

Pada bagian ini kita mengevaluasi kecenderungan sentral, pencilan (*outliers*), dan skewness dari variabel numerik utama untuk mendeteksi apakah data berdistribusi normal atau memiliki skewness ekstrem.

In [3]:
# Numerical Feature Statistics & Skewness Analysis
num_cols = ['Experience_Years', 'Daily_Token_Usage', 'Tasks_Automated_Per_Week', 'Productivity_Gain_Percent']
summary_df = df_micro[num_cols].describe().T
summary_df['skewness'] = df_micro[num_cols].skew()
summary_df['kurtosis'] = df_micro[num_cols].kurt()

print("=== STATISTICAL SUMMARY & DISTRIBUTIONS ===")
print(summary_df[['count', 'mean', 'std', 'min', '50%', 'max', 'skewness', 'kurtosis']])

=== STATISTICAL SUMMARY & DISTRIBUTIONS ===
                             count         mean          std    min     50%  \
Experience_Years           15000.0    13.013400     7.215642    1.0    13.0   
Daily_Token_Usage          15000.0  8958.312200  8449.890624  411.0  7336.0   
Tasks_Automated_Per_Week   15000.0     1.913267     1.176196    1.0     2.0   
Productivity_Gain_Percent  15000.0    11.215827    11.578100    0.3     8.1   

                               max  skewness   kurtosis  
Experience_Years              25.0  0.000240  -1.217810  
Daily_Token_Usage          58989.0  2.515177   8.148892  
Tasks_Automated_Per_Week      12.0  2.548621  11.214355  
Productivity_Gain_Percent     84.9  2.767237  10.445975  


#### 💡 Catatan Kritis Auditor Data:
1. **Right-Skewness pada Token Usage:** `Daily_Token_Usage` memiliki *skewness* positif yang signifikan, di mana nilai median (7.336 token) jauh berada di bawah rata-rata (8.958 token) dan maksimum mencapai 58.989 token. Ini mengindikasikan keberadaan kelompok *Power Users* yang mengonsumsi token secara sangat intensif.
2. **Productivity Gain Disparities:** Variabel `Productivity_Gain_Percent` memiliki rentang dari 5% hingga 84.9%. Variansi yang lebar ini memerlukan pembongkaran lebih lanjut berdasarkan kategori tools AI.

### 3. Pengujian Hipotesis Kritis & Pembongkaran Paradoks Bisnis

#### Hipotesis A: Token Usage vs Productivity Gain (Korelasi vs Diminishing Returns)
Apakah peningkatan konsumsi token secara otomatis menghasilkan kenaikan produktivitas linier?

In [4]:
# Pearson Correlation Calculation
r_token = df_micro['Daily_Token_Usage'].corr(df_micro['Productivity_Gain_Percent'])
r_tasks = df_micro['Tasks_Automated_Per_Week'].corr(df_micro['Productivity_Gain_Percent'])
r_exp = df_micro['Experience_Years'].corr(df_micro['Productivity_Gain_Percent'])

print(f"Correlation Token Usage vs Productivity Gain : r = {r_token:.4f}")
print(f"Correlation Tasks Automated vs Productivity Gain: r = {r_tasks:.4f}")
print(f"Correlation Experience Years vs Productivity Gain: r = {r_exp:.4f}")

Correlation Token Usage vs Productivity Gain : r = 0.8816
Correlation Tasks Automated vs Productivity Gain: r = 0.5548
Correlation Experience Years vs Productivity Gain: r = -0.0117


#### Hipotesis B: Jurang Efisiensi antara Specialty Tools vs Generalist Tools
Mengapa *DeepSeek* dan *GitHub Copilot* memberikan *Productivity Gain* rata-rata di atas **40%**, sementara tools seperti *ChatGPT*, *Claude*, dan *Gemini* berada di kisaran **10%**?

In [5]:
# Group analysis by Primary AI Tool
tool_perf = df_micro.groupby('Primary_AI_Tool').agg(
    Users=('User_ID', 'count'),
    Avg_Tokens=('Daily_Token_Usage', 'mean'),
    Avg_Tasks=('Tasks_Automated_Per_Week', 'mean'),
    Avg_Gain=('Productivity_Gain_Percent', 'mean'),
    Median_Gain=('Productivity_Gain_Percent', 'median')
).sort_values('Avg_Gain', ascending=False)

print("=== PERFORMANCE MATRIX BY PRIMARY AI TOOL ===")
print(tool_perf)

=== PERFORMANCE MATRIX BY PRIMARY AI TOOL ===
                    Users    Avg_Tokens  Avg_Tasks   Avg_Gain  Median_Gain
Primary_AI_Tool                                                           
DeepSeek              189  32867.010582   4.465608  42.565608         38.0
GitHub Copilot        918  32364.824619   4.127451  40.110566         36.3
Perplexity           1949   8007.466906   1.685993  10.139046          8.6
Claude (Anthropic)   2980   8040.822819   1.691275  10.119899          8.8
ChatGPT (OpenAI)     5430   8003.693923   1.679742   9.999761          8.9
Gemini (Google)      1483   7977.279838   1.694538   9.888672          8.8
Midjourney           2051   1751.994149   2.001950   2.188737          1.9


#### 🔍 Analisis Kritis Kunci:
- **Specialist Coding Tools vs Generalist Text Tools:** *GitHub Copilot* dan *DeepSeek* berfokus pada eksekusi sintaksis koding kompleks dan otomatisasi tugas teknis bernilai tinggi, sehingga menghasilkan penghematan jam kerja yang jauh lebih terukur dibanding generasi teks umum (*ChatGPT*, *Claude*, *Gemini*).
- **Kelemahan Tools Visual (Midjourney):** *Midjourney* mencatatkan *productivity gain* terendah (rata-rata 2.19%). Hal ini wajar karena pembuatan aset kreatif visual membutuhkan iterasi eksploratif dan revisi artistik manusia yang tidak serta merta memangkas durasi kerja secara otomatis.

### 4. Analisis Paritas Senioritas Pekerja (AI as a Productivity Equalizer)

Apakah AI memberikan dampak produktivitas yang lebih besar kepada pekerja Junior atau pekerja Senior?

In [6]:
# Categorize Experience
bins_exp = [-1, 3, 8, 15, 100]
labels_exp = ['Junior (0-3 yrs)', 'Mid-Level (4-8 yrs)', 'Senior (9-15 yrs)', 'Veteran (>15 yrs)']
df_micro['Experience_Group'] = pd.cut(df_micro['Experience_Years'], bins=bins_exp, labels=labels_exp)

exp_summary = df_micro.groupby('Experience_Group', observed=False).agg(
    User_Count=('User_ID', 'count'),
    Avg_Daily_Tokens=('Daily_Token_Usage', 'mean'),
    Avg_Tasks_Automated=('Tasks_Automated_Per_Week', 'mean'),
    Avg_Productivity_Gain=('Productivity_Gain_Percent', 'mean')
)

print("=== EXPERIENCE GROUP PARITY ANALYSIS ===")
print(exp_summary)

=== EXPERIENCE GROUP PARITY ANALYSIS ===
                     User_Count  Avg_Daily_Tokens  Avg_Tasks_Automated  \
Experience_Group                                                         
Junior (0-3 yrs)           1763       8885.018151             1.889393   
Mid-Level (4-8 yrs)        3055       9069.153846             1.909656   
Senior (9-15 yrs)          4129       9005.970695             1.922015   
Veteran (>15 yrs)          6053       8891.207335             1.916075   

                     Avg_Productivity_Gain  
Experience_Group                            
Junior (0-3 yrs)                 11.248780  
Mid-Level (4-8 yrs)              11.417741  
Senior (9-15 yrs)                11.274691  
Veteran (>15 yrs)                11.064167  


#### 💡 Temuan Kritis Senioritas:
Korelasi antara masa kerja (`Experience_Years`) dan `Productivity_Gain_Percent` adalah **$r = -0.0117$** (mendekati nol). Ini membuktikan bahwa **AI berfungsi sebagai alat demokratisasi produktivitas** (*great equalizer*), di mana manfaat peningkatan efisiensi dinikmati secara merata oleh pekerja pemula maupun profesional berpengalaman.

### 5. Rekomendasi Strategis & Implikasi Bisnis (Strategic Recommendations)

Berdasarkan analisis kritis di atas, berikut adalah 4 rekomendasi strategis bagi pimpinan organisasi/bisnis:

1. **Alokasi Investasi AI Terarah (Targeted AI Tooling):** Prioritaskan pengadaan tools AI terpesialisasi (*Domain-Specific AI Tools*) seperti *Copilot* dan *DeepSeek* untuk divisi teknis koding/analitis daripada hanya menyediakannya sebagai lisensi umum.
2. **Optimalisasi Konsumsi Token:** Karena korelasi token terhadap produktivitas sangat tinggi ($r = 0.88$), sediakan batas kuota token harian yang memadai agar karyawan tidak mengalami hambatan efisiensi.
3. **Standardisasi Otomatisasi Alur Kerja:** Dorong integrasi alur kerja di mana AI tidak sekadar digunakan untuk tanya-jawab (*Q&A*), tetapi digunakan untuk otomatisasi alur kerja berulang (*task automation*).
4. **Mitigasi Bias Self-Reporting:** Untuk riset internal lanjutan, disarankan menggabungkan variabel persepsi produktivitas dengan metrik kinerja objektif (*seperti Pull Request completion time, ticket closure rate, atau project turnaround time*).